# 06 — Frozen pilot-002 downstream audit
Exploratory judging of unchanged pilot-002 rubrics. Restart-safe and disabled by default.

## 1. Paths and configuration

In [ ]:
from pathlib import Path

from rubric_order.api import find_project_root, load_experiment_config

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot_002_downstream_audit.yaml"
RUN_ID = "pilot-002-downstream-audit"
RUNTIME_OVERRIDES = {}
EXECUTE_MODEL_CALLS = False
MAX_CALLS = 0
config = load_experiment_config(CONFIG_PATH, overrides=RUNTIME_OVERRIDES)
target = PROJECT_ROOT / config["output_root"] / RUN_ID

## 2. Provenance

In [ ]:
from rubric_order.downstream import plan_downstream_audit

provenance = plan_downstream_audit(config, include_requests=False)
assert (
    provenance["source_inventory"]["sha256"]
    == config["downstream_audit"]["source_content_inventory_sha256"]
)

## 3. Approved-judge gate

In [ ]:
from rubric_order.downstream import verify_approved_judge

judge_gate = verify_approved_judge(config)
assert judge_gate["status"] == "JUDGE_APPROVED"

## 4. Data and artifact integrity

In [ ]:
integrity = provenance["rubric_integrity"]
assert integrity["invalid_rubrics"] == 0
assert provenance["manual_gate"]["artifact_sha256"]

## 5. Exact estimator

In [ ]:
from rubric_order.downstream import estimate_downstream_audit

estimate = estimate_downstream_audit(config)
assert (
    estimate["updater_calls"],
    estimate["physical_judge_calls"],
    estimate["logical_predictions"],
) == (0, 500, 640)

## 6. Read-only dry-run

In [ ]:
dry_run = plan_downstream_audit(config)
assert dry_run["ready"] and dry_run["invalid_requests"] == 0
assert dry_run["fresh_requests"] == 500

## 7. Explicit execution — disabled
Only the canonical 10-call canary or 500-call resume budget is accepted here.

In [ ]:
from rubric_order.downstream import run_downstream_audit

if EXECUTE_MODEL_CALLS:
    assert MAX_CALLS in {10, 500}
    assert dry_run["ready"] and judge_gate["status"] == "JUDGE_APPROVED"
    if MAX_CALLS == 10:
        assert not target.exists(), f"target already exists: {target}"
    execution = run_downstream_audit(config, RUN_ID, execute_model_calls=True, max_calls=MAX_CALLS)
else:
    assert MAX_CALLS == 0
    execution = {"executed": False, "actual_calls": 0, "ollama_contacted": False}

## 8. Resume and status

In [ ]:
prediction_path = target / "artifacts" / "predictions.jsonl"
completed_predictions = (
    sum(1 for _ in prediction_path.open(encoding="utf-8")) if prediction_path.exists() else 0
)
resume_status = {
    "completed_logical_predictions": completed_predictions,
    "target_exists": target.exists(),
}

## 9. Analysis

In [ ]:
from rubric_order.downstream import analyze_downstream_predictions

analysis = (
    analyze_downstream_predictions(config, prediction_path, target / "metrics")
    if completed_predictions == 640
    else {"status": "awaiting_640_predictions", "exploratory": True}
)

## 10. Readiness

In [ ]:
from rubric_order.downstream import downstream_readiness

readiness = downstream_readiness(config, target)

## 11. Unload model

In [ ]:
unload_status = {
    "handled_by_execution_stage": bool(EXECUTE_MODEL_CALLS),
    "model_loaded_by_notebook": False,
}